### Task 1: Data Ingestion & Structural Audit

In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("saas_telemetry_raw.csv")

In [12]:
df.head()

,user_id,signup_date,region,subscription_tier,monthly_usage_hours,monthly_bill
0,UID_10000,2024-01-01,EMEA,Pro,32.3,33.9
1,UID_10001,2024-01-01,APAC,Basic,28.2,10.32
2,UID_10002,2024-01-02,APAC,Enterprise,284.9,311.52
3,UID_10003,2024-01-02,North America,Basic,32.4,14.87
4,UID_10004,2024-01-03,EMEA,Pro,68.5,30.99


In [13]:
print("Shape of dataset:", df.shape)

Shape of dataset: (300, 6)


In [14]:
print("\nData Types:\n")
print(df.dtypes)


Data Types:

user_id                str
signup_date            str
region                 str
subscription_tier      str
monthly_usage_hours    str
monthly_bill           str
dtype: object


In [15]:
print("\nMissing Values:\n")
print(df.isnull().sum())


Missing Values:

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    2
monthly_bill           1
dtype: int64


In [16]:
missing_flags = ["N/A", "missing", "-999"]

# Checking occurrences column-wise
for col in df.columns:
    count = df[col].isin(missing_flags).sum()
    if count > 0:
        print(f"{col}: {count} non-standard missing values")

monthly_usage_hours: 2 non-standard missing values
monthly_bill: 1 non-standard missing values


---

### Task 2: Data Cleaning & Quality Enforcement

In [17]:
df.replace(missing_flags, np.nan, inplace=True)

,user_id,signup_date,region,subscription_tier,monthly_usage_hours,monthly_bill
0,UID_10000,2024-01-01,EMEA,Pro,32.3,33.9
1,UID_10001,2024-01-01,APAC,Basic,28.2,10.32
2,UID_10002,2024-01-02,APAC,Enterprise,284.9,311.52
3,UID_10003,2024-01-02,North America,Basic,32.4,14.87
4,UID_10004,2024-01-03,EMEA,Pro,68.5,30.99
...,...,...,...,...,...,...
295,UID_10295,2024-05-27,APAC,Basic,20.7,14.67
296,UID_10296,2024-05-28,APAC,Basic,7.4,14.45
297,UID_10297,2024-05-28,EMEA,Enterprise,324.1,208.79
298,UID_10298,2024-05-29,North America,Enterprise,326.5,370.67


In [19]:
df["signup_date"] = pd.to_datetime(df["signup_date"])

In [20]:
df.head()

,user_id,signup_date,region,subscription_tier,monthly_usage_hours,monthly_bill
0,UID_10000,2024-01-01,EMEA,Pro,32.3,33.9
1,UID_10001,2024-01-01,APAC,Basic,28.2,10.32
2,UID_10002,2024-01-02,APAC,Enterprise,284.9,311.52
3,UID_10003,2024-01-02,North America,Basic,32.4,14.87
4,UID_10004,2024-01-03,EMEA,Pro,68.5,30.99


In [22]:
df["monthly_usage_hours"] = pd.to_numeric(df["monthly_usage_hours"])

In [23]:
df.loc[df["monthly_usage_hours"] < 0, "monthly_usage_hours"] = np.nan

In [24]:
print(df.dtypes)

user_id                           str
signup_date            datetime64[us]
region                            str
subscription_tier                 str
monthly_usage_hours           float64
monthly_bill                      str
dtype: object


In [26]:
df["monthly_usage_hours"] = df.groupby("subscription_tier")["monthly_usage_hours"] \
                              .transform(lambda x: x.fillna(x.median()))

In [27]:
print("\nMissing values after cleaning:\n")
print(df.isnull().sum())

print("\nData types after cleaning:\n")
print(df.dtypes)


Missing values after cleaning:

user_id                0
signup_date            0
region                 0
subscription_tier      0
monthly_usage_hours    0
monthly_bill           2
dtype: int64

Data types after cleaning:

user_id                           str
signup_date            datetime64[us]
region                            str
subscription_tier                 str
monthly_usage_hours           float64
monthly_bill                      str
dtype: object


---

### Task 3: Feature Engineering with Group Transformations

In [28]:
df["tier_avg_usage"] = df.groupby("subscription_tier")["monthly_usage_hours"] \
                          .transform("mean")

In [29]:
df["usage_deviation"] = df["monthly_usage_hours"] - df["tier_avg_usage"]

In [30]:
df["tier_avg_usage"] = df["tier_avg_usage"].round(2)
df["usage_deviation"] = df["usage_deviation"].round(2)

In [31]:
df[["subscription_tier", "monthly_usage_hours", "tier_avg_usage", "usage_deviation"]].head()

,subscription_tier,monthly_usage_hours,tier_avg_usage,usage_deviation
0,Pro,32.3,73.74,-41.44
1,Basic,28.2,22.91,5.29
2,Enterprise,284.9,232.38,52.52
3,Basic,32.4,22.91,9.49
4,Pro,68.5,73.74,-5.24


---

### Task 4: Executive Aggregations (Split-Apply-Combine)

In [34]:
df["monthly_bill"] = pd.to_numeric(df["monthly_bill"])

In [35]:
executive_summary = df.groupby(["subscription_tier", "region"]).agg(
    total_users=("user_id", "nunique"),
    avg_monthly_bill=("monthly_bill", "mean"),
    total_usage_hours=("monthly_usage_hours", "sum"),
    max_usage_deviation=("usage_deviation", "max")
).reset_index()

In [36]:
executive_summary["avg_monthly_bill"] = executive_summary["avg_monthly_bill"].round(2)
executive_summary["total_usage_hours"] = executive_summary["total_usage_hours"].round(2)
executive_summary["max_usage_deviation"] = executive_summary["max_usage_deviation"].round(2)

In [37]:
executive_summary.head()

,subscription_tier,region,total_users,avg_monthly_bill,total_usage_hours,max_usage_deviation
0,Basic,APAC,32,12.35,706.05,16.29
1,Basic,EMEA,37,12.68,836.70,16.99
2,Basic,LATAM,44,12.57,989.90,16.19
3,Basic,North America,33,12.92,812.35,16.49
4,Enterprise,APAC,11,390.78,2509.30,87.12


---

### Task 5

In [38]:
pivot_table = pd.pivot_table(
    df,
    index="subscription_tier",     
    columns="region",             
    values="monthly_bill",        
    aggfunc="mean",              
    margins=True                  
)

In [39]:
pivot_table = pivot_table.round(2)

In [40]:
pivot_table

region,APAC,EMEA,LATAM,North America,All
subscription_tier,,,,,
Basic,12.35,12.68,12.57,12.92,12.63
Enterprise,390.78,315.53,345.00,405.70,363.58
Pro,39.59,38.62,41.22,40.77,40.10
All,87.84,69.31,64.61,78.06,74.40
